# Visual Embedding

In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 3050 Laptop GPU


In [2]:
import torch
import numpy as np
from PIL import Image
from pathlib import Path
from tqdm import tqdm
import pickle
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import clip

In [3]:
clip.available_models()

['RN50',
 'RN101',
 'RN50x4',
 'RN50x16',
 'RN50x64',
 'ViT-B/32',
 'ViT-B/16',
 'ViT-L/14',
 'ViT-L/14@336px']

# Vit32 and Clip

In [4]:
# ============================================================================
# CONFIGURATION
# ============================================================================
class Config:
    # Paths
    TRAIN_IMAGE_DIR = "../train_images"
    TEST_IMAGE_DIR = "../test_images"
    TRAIN_OUTPUT_PATH = "train_embeddings_clip.npy"
    TEST_OUTPUT_PATH = "test_embeddings_clip.npy"
    TRAIN_IDS_PATH = "train_image_ids_clip.pkl"
    TEST_IDS_PATH = "test_image_ids_clip.pkl"
    FAILED_IMAGES_PATH = "failed_images.txt"
    
    # Model settings
    MODEL_NAME = "ViT-B/32"  # Options: "ViT-B/32", "ViT-B/16", "ViT-L/14"
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    BATCH_SIZE = 128  # Adjust based on GPU memory (256 for 16GB, 128 for 8GB)
    NUM_WORKERS = 4
    
    # Processing
    IMAGE_SIZE = 224  # CLIP default
    NORMALIZE = True
    
    # Download settings
    AMAZON_IMAGE_URL_TEMPLATE = "https://m.media-amazon.com/images/I/{}"
    DOWNLOAD_TIMEOUT = 10  # seconds


# ============================================================================
# IMAGE DOWNLOAD UTILITY
# ============================================================================
import requests
from io import BytesIO

def download_image_from_amazon(image_code, timeout=10):
    """
    Download image from Amazon using the image code
    
    Args:
        image_code: The image filename (e.g., '318UdBSxHrL.jpg')
        timeout: Request timeout in seconds
    
    Returns:
        PIL Image object or None if download fails
    """
    url = f"https://m.media-amazon.com/images/I/{image_code}"
    
    try:
        response = requests.get(url, timeout=timeout, headers={
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        })
        
        if response.status_code == 200:
            image = Image.open(BytesIO(response.content)).convert('RGB')
            return image
        else:
            return None
            
    except Exception as e:
        return None


def save_downloaded_image(image, save_path):
    """Save downloaded image to disk"""
    try:
        image.save(save_path, quality=95)
        return True
    except Exception as e:
        print(f"Error saving image to {save_path}: {e}")
        return False


# ============================================================================
# DATASET CLASS
# ============================================================================
class ImageDataset(Dataset):
    """Efficient dataset for loading images with auto-download fallback"""
    
    def __init__(self, image_dir, transform=None, config=None):
        self.image_dir = Path(image_dir)
        self.image_paths = sorted(list(self.image_dir.glob("*.jpg")) + 
                                 list(self.image_dir.glob("*.png")) +
                                 list(self.image_dir.glob("*.jpeg")))
        self.transform = transform
        self.config = config
        self.failed_images = []  # Track images that fail even after redownload
        
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        img_id = img_path.stem
        image_filename = img_path.name
        
        try:
            # First attempt: Load from disk
            image = Image.open(img_path).convert('RGB')
            
            if self.transform:
                image = self.transform(image)
            
            return image, img_id
            
        except Exception as e:
            # First load failed, try downloading from Amazon
            print(f"\nError loading {img_path.name}: {str(e)[:50]}")
            print(f"Attempting to download from Amazon...")
            
            downloaded_image = download_image_from_amazon(
                image_filename, 
                timeout=self.config.DOWNLOAD_TIMEOUT if self.config else 10
            )
            
            if downloaded_image is not None:
                print(f"✓ Successfully downloaded {image_filename}")
                
                # Save the downloaded image to disk for future use
                save_downloaded_image(downloaded_image, img_path)
                
                # Transform and return
                if self.transform:
                    downloaded_image = self.transform(downloaded_image)
                
                return downloaded_image, img_id
            else:
                # Download also failed
                print(f"✗ Failed to download {image_filename}")
                self.failed_images.append(image_filename)
                
                # Return a blank image as fallback
                blank = Image.new('RGB', (224, 224), (255, 255, 255))
                if self.transform:
                    blank = self.transform(blank)
                return blank, img_id


# ============================================================================
# EMBEDDING EXTRACTOR
# ============================================================================
class EmbeddingExtractor:
    """Extract embeddings using CLIP"""
    
    def __init__(self, config):
        self.config = config
        self.device = torch.device(config.DEVICE)
        self.all_failed_images = []  # Track all failed images across datasets
        
        print(f"Loading CLIP model: {config.MODEL_NAME}")
        print(f"Device: {self.device}")
        
        # Load CLIP model
        self.model, self.preprocess = clip.load(config.MODEL_NAME, device=self.device)
        self.model.eval()
        
        print(f"Model loaded successfully")
        print(f"Embedding dimension: {self.model.visual.output_dim}")
    
    def extract_embeddings(self, image_dir, output_path, ids_path):
        """Extract and save embeddings from images"""
        
        print(f"\nProcessing images from: {image_dir}")
        
        # Create dataset and dataloader
        dataset = ImageDataset(image_dir, transform=self.preprocess, config=self.config)
        dataloader = DataLoader(
            dataset,
            batch_size=self.config.BATCH_SIZE,
            shuffle=False,
            num_workers=self.config.NUM_WORKERS,
            pin_memory=True if self.config.DEVICE == "cuda" else False
        )
        
        print(f"Total images: {len(dataset)}")
        print(f"Batch size: {self.config.BATCH_SIZE}")
        
        all_embeddings = []
        all_ids = []
        
        # Extract embeddings
        with torch.no_grad():
            for images, img_ids in tqdm(dataloader, desc="Extracting embeddings"):
                images = images.to(self.device)
                
                # Get image features from CLIP
                features = self.model.encode_image(images)
                
                # Normalize embeddings (important for downstream tasks)
                features = features / features.norm(dim=-1, keepdim=True)
                
                # Move to CPU and convert to numpy
                embeddings = features.cpu().numpy()
                
                all_embeddings.append(embeddings)
                all_ids.extend(img_ids)
        
        # Concatenate all embeddings
        all_embeddings = np.vstack(all_embeddings)
        
        # Track failed images from this dataset
        self.all_failed_images.extend(dataset.failed_images)
        
        print(f"\nEmbeddings shape: {all_embeddings.shape}")
        print(f"Total IDs: {len(all_ids)}")
        print(f"Failed images in this dataset: {len(dataset.failed_images)}")
        
        # Save embeddings and IDs
        np.save(output_path, all_embeddings)
        with open(ids_path, 'wb') as f:
            pickle.dump(all_ids, f)
        
        print(f"Saved embeddings to: {output_path}")
        print(f"Saved IDs to: {ids_path}")
        
        return all_embeddings, all_ids


# ============================================================================
# ALTERNATIVE: Using ResNet50 (faster but less semantic)
# ============================================================================
class ResNetExtractor:
    """Alternative: Extract embeddings using ResNet50"""
    
    def __init__(self, config):
        self.config = config
        self.device = torch.device(config.DEVICE)
        self.all_failed_images = []
        
        from torchvision.models import resnet50, ResNet50_Weights
        
        print(f"Loading ResNet50 model")
        print(f"Device: {self.device}")
        
        # Load pre-trained ResNet50
        self.model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        # Remove final classification layer to get embeddings
        self.model = torch.nn.Sequential(*list(self.model.children())[:-1])
        self.model = self.model.to(self.device)
        self.model.eval()
        
        # Define preprocessing
        self.preprocess = transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                               std=[0.229, 0.224, 0.225])
        ])
        
        print("ResNet50 loaded (2048-dim embeddings)")
    
    def extract_embeddings(self, image_dir, output_path, ids_path):
        """Extract and save embeddings"""
        
        print(f"\nProcessing images from: {image_dir}")
        
        dataset = ImageDataset(image_dir, transform=self.preprocess, config=self.config)
        dataloader = DataLoader(
            dataset,
            batch_size=self.config.BATCH_SIZE,
            shuffle=False,
            num_workers=self.config.NUM_WORKERS,
            pin_memory=True if self.config.DEVICE == "cuda" else False
        )
        
        print(f"Total images: {len(dataset)}")
        print(f"Batch size: {self.config.BATCH_SIZE}")
        
        all_embeddings = []
        all_ids = []
        
        with torch.no_grad():
            for images, img_ids in tqdm(dataloader, desc="Extracting embeddings"):
                images = images.to(self.device)
                features = self.model(images)
                features = features.squeeze(-1).squeeze(-1)  # Remove spatial dims
                
                embeddings = features.cpu().numpy()
                all_embeddings.append(embeddings)
                all_ids.extend(img_ids)
        
        all_embeddings = np.vstack(all_embeddings)
        
        # Track failed images
        self.all_failed_images.extend(dataset.failed_images)
        
        print(f"\nEmbeddings shape: {all_embeddings.shape}")
        print(f"Total IDs: {len(all_ids)}")
        print(f"Failed images in this dataset: {len(dataset.failed_images)}")
        
        np.save(output_path, all_embeddings)
        with open(ids_path, 'wb') as f:
            pickle.dump(all_ids, f)
        
        print(f"Saved embeddings to: {output_path}")
        print(f"Saved IDs to: {ids_path}")
        
        return all_embeddings, all_ids


# ============================================================================
# MAIN EXECUTION
# ============================================================================
def main():
    config = Config()
    
    # Choose extractor: CLIP (recommended) or ResNet50 (faster)
    extractor = EmbeddingExtractor(config)
    # extractor = ResNetExtractor(config)  # Using ResNet as per your output
    
    # Extract training embeddings
    print("="*80)
    print("EXTRACTING TRAINING EMBEDDINGS")
    print("="*80)
    train_embeddings, train_ids = extractor.extract_embeddings(
        config.TRAIN_IMAGE_DIR,
        config.TRAIN_OUTPUT_PATH,
        config.TRAIN_IDS_PATH
    )
    
    # Extract test embeddings
    print("\n" + "="*80)
    print("EXTRACTING TEST EMBEDDINGS")
    print("="*80)
    test_embeddings, test_ids = extractor.extract_embeddings(
        config.TEST_IMAGE_DIR,
        config.TEST_OUTPUT_PATH,
        config.TEST_IDS_PATH
    )
    
    # Save list of failed images
    if extractor.all_failed_images:
        with open(config.FAILED_IMAGES_PATH, 'w') as f:
            for img in extractor.all_failed_images:
                f.write(f"{img}\n")
        
        print(f"\n⚠ WARNING: {len(extractor.all_failed_images)} images failed even after redownload attempt")
        print(f"Failed images saved to: {config.FAILED_IMAGES_PATH}")
        print("\nFailed images:")
        for img in extractor.all_failed_images:
            print(f"  - {img}")
    else:
        print("\n✓ All images processed successfully!")
    
    print("\n" + "="*80)
    print("EXTRACTION COMPLETE")
    print("="*80)
    print(f"Training embeddings: {train_embeddings.shape}")
    print(f"Test embeddings: {test_embeddings.shape}")
    print(f"\nFiles saved:")
    print(f"  - {config.TRAIN_OUTPUT_PATH}")
    print(f"  - {config.TEST_OUTPUT_PATH}")
    print(f"  - {config.TRAIN_IDS_PATH}")
    print(f"  - {config.TEST_IDS_PATH}")
    if extractor.all_failed_images:
        print(f"  - {config.FAILED_IMAGES_PATH}")


# ============================================================================
# UTILITY: Load embeddings for training
# ============================================================================
def load_embeddings(embeddings_path, ids_path):
    """Utility function to load saved embeddings"""
    embeddings = np.load(embeddings_path)
    with open(ids_path, 'rb') as f:
        ids = pickle.load(f)
    return embeddings, ids


if __name__ == "__main__":
    # Install required packages first:
    # pip install torch torchvision clip-by-openai Pillow numpy tqdm requests
    # For CLIP: pip install git+https://github.com/openai/CLIP.git
    
    main()
    
    # Example: Loading embeddings later for regression
    # train_emb, train_ids = load_embeddings("train_embeddings.npy", "train_image_ids.pkl")
    # test_emb, test_ids = load_embeddings("test_embeddings.npy", "test_image_ids.pkl")

Loading CLIP model: ViT-B/32
Device: cuda
Model loaded successfully
Embedding dimension: 512
EXTRACTING TRAINING EMBEDDINGS

Processing images from: ../train_images
Total images: 72287
Batch size: 128


Extracting embeddings: 100%|██████████| 565/565 [29:18<00:00,  3.11s/it]



Embeddings shape: (72287, 512)
Total IDs: 72287
Failed images in this dataset: 0
Saved embeddings to: train_embeddings_clip.npy
Saved IDs to: train_image_ids_clip.pkl

EXTRACTING TEST EMBEDDINGS

Processing images from: ../test_images
Total images: 72221
Batch size: 128


Extracting embeddings: 100%|██████████| 565/565 [28:55<00:00,  3.07s/it]



Embeddings shape: (72221, 512)
Total IDs: 72221
Failed images in this dataset: 0
Saved embeddings to: test_embeddings_clip.npy
Saved IDs to: test_image_ids_clip.pkl

✓ All images processed successfully!

EXTRACTION COMPLETE
Training embeddings: (72287, 512)
Test embeddings: (72221, 512)

Files saved:
  - train_embeddings_clip.npy
  - test_embeddings_clip.npy
  - train_image_ids_clip.pkl
  - test_image_ids_clip.pkl


# VitL34 / Dinov2 Embeddings extraction

In [4]:
import os
import torch
import numpy as np
from tqdm import tqdm
from PIL import Image
from transformers import AutoModel, AutoImageProcessor
from torch.utils.data import DataLoader, Dataset

# ============================================================
# CONFIG
# ============================================================
class CFG:
    MODEL_NAME = "facebook/dinov2-large"
    IMAGE_DIR = "train_images"        # path to your image folder
    OUTPUT_PATH = "train_dinov2_emb.npy"
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    BATCH_SIZE = 2                    # safe for 4GB GPU
    NUM_WORKERS = 2

# ============================================================
# DATASET
# ============================================================
class ImageDataset(Dataset):
    def __init__(self, image_dir, processor):
        self.image_dir = image_dir
        self.image_files = sorted(os.listdir(image_dir))
        self.processor = processor

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        fname = self.image_files[idx]
        fpath = os.path.join(self.image_dir, fname)
        try:
            image = Image.open(fpath).convert("RGB")
            inputs = self.processor(images=image, return_tensors="pt")
            return inputs, fname
        except Exception:
            return None, fname  # failed image

# ============================================================
# EXTRACTION FUNCTION
# ============================================================
def extract_dinov2_embeddings():
    print(f"🔹 Loading model: {CFG.MODEL_NAME}")
    processor = AutoImageProcessor.from_pretrained(CFG.MODEL_NAME)
    model = AutoModel.from_pretrained(CFG.MODEL_NAME).to(CFG.DEVICE)
    model.eval()

    dataset = ImageDataset(CFG.IMAGE_DIR, processor)
    dataloader = DataLoader(dataset, batch_size=CFG.BATCH_SIZE,
                            shuffle=False, num_workers=CFG.NUM_WORKERS,
                            pin_memory=True, collate_fn=lambda x: [y for y in x if y[0] is not None])

    features, ids, failed = [], [], []

    with torch.no_grad():
        for batch in tqdm(dataloader, total=len(dataloader)):
            if len(batch) == 0:
                continue
            try:
                imgs = {k: torch.cat([b[0][k] for b in batch]).to(CFG.DEVICE, non_blocking=True)
                        for k in batch[0][0].keys()}
                fnames = [b[1] for b in batch]

                with torch.cuda.amp.autocast():
                    outputs = model(**imgs)
                    # Mean pool CLS tokens
                    emb = outputs.last_hidden_state.mean(dim=1)
                    emb = emb / emb.norm(dim=-1, keepdim=True)
                    features.append(emb.cpu().numpy())
                    ids.extend(fnames)

                torch.cuda.empty_cache()
            except RuntimeError as e:
                if "CUDA out of memory" in str(e):
                    print("⚠️ OOM - retrying each image individually...")
                    torch.cuda.empty_cache()
                    for b in batch:
                        try:
                            inputs = {k: v.to(CFG.DEVICE) for k, v in b[0].items()}
                            with torch.cuda.amp.autocast():
                                out = model(**inputs)
                                f = out.last_hidden_state.mean(dim=1)
                                f = f / f.norm(dim=-1, keepdim=True)
                                features.append(f.cpu().numpy())
                                ids.append(b[1])
                        except Exception:
                            failed.append(b[1])
                            continue
                        finally:
                            torch.cuda.empty_cache()
                else:
                    failed.extend([b[1] for b in batch])

    features = np.concatenate(features, axis=0)
    np.save(CFG.OUTPUT_PATH, features)
    print(f"✅ Embeddings saved to {CFG.OUTPUT_PATH}")
    print(f"⚠️ Failed images: {len(failed)}")

    return features, ids, failed

# ============================================================
# MAIN
# ============================================================
if __name__ == "__main__":
    feats, ids, failed = extract_dinov2_embeddings()
    print("Features shape:", feats.shape)


🔹 Loading model: facebook/dinov2-large


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 3.81 GiB of which 15.69 MiB is free. Including non-PyTorch memory, this process has 3.29 GiB memory in use. Of the allocated memory 3.14 GiB is allocated by PyTorch, and 58.12 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)